In [94]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# This is for the graph visualization
from graphviz import Digraph

In [95]:
# starting the Project
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self._prev = set(_children)
        self.data = data
        self._op = _op
        self.grad = 0.0
        self._backward = lambda: None
        self.label = label
        
    def __repr__(self):
        return f"Value(data = {self.data})"
        
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
            
        out._backward = _backward
        return out
    def __neg__(self):
        return -1 * self
    def __radd__(self, other): # other + self
        return self + other
    def __sub__(self,other):
        return self + (-other)
        
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
            
        out._backward = _backward
        return out
    def __rmul__(self, other):
        return self * other
    def __pow__(self,other):
        assert isinstance(other,(int,float)),f"We Need Only Numbers(int/float) Fella :)"
        out = Value(self.data ** other,(self,),f'**{other}')
        def _backward():
            self.grad += other * (self.data **(other-1)) * out.grad
        out._backward = _backward
        return out
    def __truediv__(self,other):
        return self * other**-1
    def exp(self):
        x = self.data
        out = Value(math.exp(x),(self,),'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        
        def _backward():
            self.grad += (1 - t**2) * out.grad
            
        out._backward = _backward
        return out
        
    # Define backward the Recursive Function
    def backward(self):
        # Create empty list topo and the empty set visited
        topo = []
        visited = set()
        
        # Create Build TOPO function to append to it the children
        def build_topo(v):
            # check if we already looped over v
            if v not in visited:
                # I should add it for the next time
                visited.add(v)
                # Recursion Time
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
                
        # NOW WE BUILD THE TOPO OF THE OUTPUT LAYER
        build_topo(self)
        
        self.grad = 1.0 # THE DERIVATIVE OF THE OUTPUT LAYER WITH RESPECT TO ITSELF IS ALWAYS 1
        
        for node in reversed(topo): # we use Reversed because the output layer will be in the end of topo but we need to start with it
            node._backward()   # Now We calculate the grad from the end to start

In [96]:
from graphviz import Digraph

def trace(root):
    # Builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # Create a rectangular ('record') node for every Value in the graph
        dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data,n.grad), shape='record')

        if n._op:
            # If this value is a result of an operation (+ or *), create an op node for it
            dot.node(name=uid + n._op, label=n._op)
            # Connect the op node to the actual Value node
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # Connect the child node to the op node of its parent
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [97]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        # dot product + bias
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]
        
class Layer:
    def __init__(self,nin,nout):
        #List of nout neurons , each of them expect nin inputs
        self.neurons = [Neuron(nin) for _ in range(nout)] 
    def __call__(self,x):
        #pass the inputs to each neuron in this layer
        outs = [n(x) for n in self.neurons]
        return outs[0]if len(outs) == 1 else outs 

    def parameters(self):
        #create a list of all parameters in the layer by looping over all the parameters in each neuron
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    def __init__(self,nin,nouts):
        #Adjust the list of nouts(which is the number of neurons in each layer) to start with the number of inputs
        sz = [nin] + nouts
        #Make pairs of each layer and the output of the layer before it which is going to be its input basically
        self.layers = [Layer(sz[i],sz[i+1]) for i in range(len(nouts))]

    def __call__(self,x):
        #for each layer in the different layers
        for layer in self.layers:
            #overrite the inputs with the output of this layer so it be the input of the next layer
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters() ] 


In [98]:
n = MLP(3,[4,4,1])

In [99]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0]

In [100]:
for k in range(20):
    #Forward Pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for yout, ygt in zip(ypred, ys))

    #zero the parameters
    for p in n.parameters():
        p.grad = 0.0
    #Backward Pass
    loss.backward()

    #Updating
    for p in n.parameters():
        p.data += -0.1 * p.grad
    print(k,loss.data)

0 4.5002835505873815
1 1.6005940555879334
2 0.03594266200956173
3 0.031042160163886945
4 0.02739548903863644
5 0.024550409941071185
6 0.022257984597181996
7 0.020366236857668432
8 0.01877579640786399
9 0.017418401169009265
10 0.016245360363400442
11 0.015220889029726974
12 0.014318034964969585
13 0.013516077194633722
14 0.012798803426305665
15 0.012153334778602826
16 0.011569303222918933
17 0.011038263096480552
18 0.010553261918394438
19 0.010108522034573296


In [101]:
ypred

[Value(data = 0.9509793677807622),
 Value(data = -0.9565487243747521),
 Value(data = -0.9497098190546651),
 Value(data = 0.9426555669796646)]